In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
import numpy as np

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
from pathlib import Path

for path in Path("/kaggle/input/datasets/timjas/archive").iterdir():
    print(path)

/kaggle/input/datasets/timjas/archive/val
/kaggle/input/datasets/timjas/archive/train


In [6]:
ROOT = "/kaggle/input/datasets/timjas/archive"


確認 Kaggle 的資料

In [7]:
from pathlib import Path

root = Path(ROOT)

print("Train:", (root / "train").exists())
print("Val:", (root / "val").exists())

routes = [
    p for p in (root / "train").iterdir()
    if p.is_dir()
]

print("Number of train routes:", len(routes))
print("First route:", routes[0])


Train: True
Val: True
Number of train routes: 20
First route: /kaggle/input/datasets/timjas/archive/train/routes_30mshortroutes_Town05_Scenario8junction_route1_11_23_20_29_34


剛才成功的 LiDAR preprocessing

In [10]:
import numpy as np
import torch


def lidar_to_bev(
    lidar,
    x_range=(-32, 32),
    y_range=(-32, 32),
    z_range=(-3, 5),
    resolution=0.25
):
    x = lidar[:, 0]
    y = lidar[:, 1]
    z = lidar[:, 2]
    intensity = lidar[:, 3]

    mask = (
        (x >= x_range[0]) & (x < x_range[1]) &
        (y >= y_range[0]) & (y < y_range[1]) &
        (z >= z_range[0]) & (z < z_range[1])
    )

    x = x[mask]
    y = y[mask]
    z = z[mask]
    intensity = intensity[mask]

    H = int((x_range[1] - x_range[0]) / resolution)
    W = int((y_range[1] - y_range[0]) / resolution)

    bev = np.zeros(
        (3, H, W),
        dtype=np.float32
    )

    xi = ((x - x_range[0]) / resolution).astype(np.int32)
    yi = ((y - y_range[0]) / resolution).astype(np.int32)

    height = (
        (z - z_range[0])
        / (z_range[1] - z_range[0])
    )

    height = np.clip(height, 0, 1)

    for i in range(len(x)):

        xx = xi[i]
        yy = yi[i]

        bev[0, yy, xx] = max(
            bev[0, yy, xx],
            height[i]
        )

        bev[1, yy, xx] = max(
            bev[1, yy, xx],
            intensity[i]
        )

        bev[2, yy, xx] += 1

    bev[2] = np.clip(
        bev[2] / 10.0,
        0,
        1
    )

    return torch.from_numpy(bev)


6. 建立 Dataset

In [12]:
from pathlib import Path
import json

import numpy as np
import torch

from PIL import Image
from torch.utils.data import Dataset


class TransFuserDataset(Dataset):

    def __init__(
        self,
        root_dir,
        split="train"
    ):

        self.root_dir = Path(root_dir)
        self.data_dir = self.root_dir / split

        self.samples = []

        for route_dir in sorted(
            self.data_dir.iterdir()
        ):

            if not route_dir.is_dir():
                continue

            rgb_dir = route_dir / "rgb"
            lidar_dir = route_dir / "lidar"
            measurement_dir = route_dir / "measurements"

            if not (
                rgb_dir.exists()
                and lidar_dir.exists()
                and measurement_dir.exists()
            ):
                continue

            for json_file in sorted(
                measurement_dir.glob("*.json")
            ):

                frame = json_file.stem

                rgb_file = (
                    rgb_dir / f"{frame}.png"
                )

                lidar_file = (
                    lidar_dir / f"{frame}.npy"
                )

                if (
                    rgb_file.exists()
                    and lidar_file.exists()
                ):

                    self.samples.append({
                        "rgb": rgb_file,
                        "lidar": lidar_file,
                        "measurement": json_file
                    })

        print(
            f"{split}: "
            f"{len(self.samples)} samples"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        # ----------------
        # RGB
        # ----------------

        image = Image.open(
            sample["rgb"]
        ).convert("RGB")

        image = np.array(image)

        image = torch.from_numpy(
            image
        ).permute(2, 0, 1)

        image = image.float() / 255.0

        # ----------------
        # LiDAR
        # ----------------

        lidar_data = np.load(
            sample["lidar"],
            allow_pickle=True
        )

        # [0] = number of points
        # [1] = actual point cloud

        lidar = lidar_data[1]

        lidar = lidar_to_bev(
            lidar
        )

        # ----------------
        # Measurement
        # ----------------

        with open(
            sample["measurement"],
            "r"
        ) as f:

            measurement = json.load(f)

        steer = float(
            measurement["steer"]
        )

        throttle = float(
            measurement["throttle"]
        )

        brake = float(
            measurement["brake"]
        )

        target = torch.tensor(
            [
                steer,
                throttle,
                brake
            ],
            dtype=torch.float32
        )

        return {
            "rgb": image,
            "lidar": lidar,
            "target": target
        }


In [13]:
train_dataset = TransFuserDataset(
    ROOT,
    split="train"
)

val_dataset = TransFuserDataset(
    ROOT,
    split="val"
)


train: 2134 samples
val: 1963 samples


7. 建立 train / validation

In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


9. 測試一個 batch

In [16]:
batch = next(iter(train_loader))

print("RGB:", batch["rgb"].shape)
print("LiDAR:", batch["lidar"].shape)
print("Target:", batch["target"].shape)


RGB: torch.Size([8, 3, 160, 960])
LiDAR: torch.Size([8, 3, 256, 256])
Target: torch.Size([8, 3])


10. 把 batch 放到 GPU

In [17]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)


cuda


In [18]:
rgb = batch["rgb"].to(
    device,
    non_blocking=True
)

lidar = batch["lidar"].to(
    device,
    non_blocking=True
)

target = batch["target"].to(
    device,
    non_blocking=True
)

print(rgb.device)
print(lidar.device)
print(target.device)


cuda:0
cuda:0
cuda:0


Kaggle
  │
  ├── GPU ✓
  │
  ├── Dataset ✓
  │
  ├── RGB ✓
  │
  ├── LiDAR → BEV ✓
  │
  ├── DataLoader
  │
  └── CUDA
        ↓
     下一步
        ↓
  RGB Encoder
  +
  LiDAR Encoder
        ↓
     Fusion
        ↓
  steer/throttle/brake

RGB [3,160,960]
      ↓
 CNN
      ↓
 Image feature
      \
       → Fusion → Control
      /
LiDAR [3,256,256]
      ↓
 CNN
      ↓
 LiDAR feature


In [19]:
import torch
import torch.nn as nn


class SimpleTransFuser(nn.Module):

    def __init__(self):
        super().__init__()

        # =========================
        # RGB Encoder
        # =========================

        self.rgb_encoder = nn.Sequential(

            nn.Conv2d(
                3, 32,
                kernel_size=5,
                stride=2,
                padding=2
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(
                64, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )


        # =========================
        # LiDAR Encoder
        # =========================

        self.lidar_encoder = nn.Sequential(

            nn.Conv2d(
                3, 32,
                kernel_size=5,
                stride=2,
                padding=2
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(
                64, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )


        # =========================
        # Fusion
        # =========================

        self.fusion = nn.Sequential(

            nn.Linear(
                128 + 128,
                256
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU()
        )


        # =========================
        # Control Head
        # =========================

        self.control_head = nn.Linear(
            128,
            3
        )


    def forward(self, rgb, lidar):

        # RGB
        rgb_feature = self.rgb_encoder(rgb)

        rgb_feature = rgb_feature.flatten(1)


        # LiDAR
        lidar_feature = self.lidar_encoder(lidar)

        lidar_feature = lidar_feature.flatten(1)


        # Fusion
        feature = torch.cat(
            [
                rgb_feature,
                lidar_feature
            ],
            dim=1
        )

        feature = self.fusion(feature)


        # Control
        control = self.control_head(
            feature
        )

        return control


把 Model 放到 GPU

In [20]:
model = SimpleTransFuser().to(device)

print(model)


SimpleTransFuser(
  (rgb_encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (lidar_encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running

4. 測試 Forward

In [21]:
with torch.no_grad():

    prediction = model(
        rgb,
        lidar
    )

print("Prediction shape:")
print(prediction.shape)

print("\nPrediction:")
print(prediction)


Prediction shape:
torch.Size([8, 3])

Prediction:
tensor([[-0.2031, -0.0645,  0.0445],
        [-0.1410, -0.0729,  0.0587],
        [-0.1536, -0.0773,  0.0457],
        [-0.1494, -0.0803,  0.0348],
        [-0.1880, -0.0663,  0.0386],
        [-0.1554, -0.0758,  0.0589],
        [-0.1681, -0.0745,  0.0793],
        [-0.2161, -0.1152,  0.0691]], device='cuda:0')


應該得到：

Prediction shape:
torch.Size([8, 3])

也就是：

8 samples
   ↓
┌───────────────┐
│ steer         │
│ throttle      │
│ brake         │
└───────────────┘

In [23]:
criterion = nn.MSELoss()

In [24]:
loss = criterion(
    prediction,
    target
)

print("Loss:", loss.item())

Loss: 0.20155876874923706


In [25]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)


In [26]:
model.train()

optimizer.zero_grad()

prediction = model(
    rgb,
    lidar
)

loss = criterion(
    prediction,
    target
)

loss.backward()

optimizer.step()

print("Loss:", loss.item())


Loss: 0.20155876874923706


這段如果成功，就代表你已經完成：

Dataset
   ↓
DataLoader
   ↓
GPU
   ↓
RGB Encoder
   +
LiDAR Encoder
   ↓
Fusion
   ↓
Prediction
   ↓
Loss
   ↓
Backpropagation
   ↓
Optimizer

這就是一個真正可以訓練的自駕模型。

In [27]:
prediction = model(rgb, lidar)

print(prediction.shape)


torch.Size([8, 3])


In [28]:
loss = criterion(prediction, target)

print(loss.item())


0.19143006205558777


In [29]:
optimizer.zero_grad()

loss.backward()

optimizer.step()

print("training step OK")


training step OK


什麼不能只 concat？
我們剛才做的是：

RGB
 ↓
CNN
 ↓
一個 feature vector ──┐
                      ├─ concat → MLP → control
LiDAR                 │
 ↓                    │
CNN                   ─┘
 ↓
一個 feature vector

RGB 和 LiDAR 在空間上的關係去哪裡了？

它沒有明確建立：

「RGB 的這個位置」和「LiDAR 的那個位置」之間的關係。

這就是 Transformer 可以幫忙的地方。

2. Transformer 到底在做什麼？
Transformer 最重要的概念之一是：

Attention

簡化來說：

這個 feature
   ↓
「我應該注意其他哪些 feature？」

例如我們把影像切成很多 feature：

RGB feature tokens

[道路] [車子] [紅綠燈] [天空] [行人] ...

LiDAR 也有：

LiDAR feature tokens

[道路] [物體] [物體] [路邊] [車輛] ...

Transformer 可以讓它們互相「看」。

我們先做一個簡化版 TransFuser
我們現在不要直接上官方完整架構。

我們做：

RGB
 ↓
CNN
 ↓
Feature Map
 ↓
Flatten
 ↓
RGB Tokens
       \
        Transformer
       /
LiDAR Tokens
 ↓
Control

In [30]:
class CNNEncoder(nn.Module):

    def __init__(self, in_channels=3):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(
                in_channels,
                32,
                kernel_size=5,
                stride=2,
                padding=2
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.encoder(x)


In [32]:
encoder_rgb = CNNEncoder(3).to(device)
encoder_lidar = CNNEncoder(3).to(device)


In [33]:
with torch.no_grad():

    rgb_feature = encoder_rgb(rgb)
    lidar_feature = encoder_lidar(lidar)

print("RGB feature:", rgb_feature.shape)
print("LiDAR feature:", lidar_feature.shape)


RGB feature: torch.Size([8, 128, 20, 120])
LiDAR feature: torch.Size([8, 128, 32, 32])


把 Feature Map 變成 Tokens
Transformer 通常吃：

[B, sequence_length, embedding_dimension]

但 CNN 給我們：

[B, C, H, W]

所以需要轉換。

def feature_to_tokens(feature):
...

In [34]:
def feature_to_tokens(feature):

    # [B, C, H, W]
    B, C, H, W = feature.shape

    # [B, C, H*W]
    feature = feature.flatten(2)

    # [B, H*W, C]
    feature = feature.transpose(1, 2)

    return feature


In [36]:
rgb_tokens = feature_to_tokens(
    rgb_feature
)

lidar_tokens = feature_to_tokens(
    lidar_feature
)

print("RGB tokens:", rgb_tokens.shape)
print("LiDAR tokens:", lidar_tokens.shape)


RGB tokens: torch.Size([8, 2400, 128])
LiDAR tokens: torch.Size([8, 1024, 128])


In [37]:
tokens = torch.cat(
    [
        rgb_tokens,
        lidar_tokens
    ],
    dim=1
)

print(tokens.shape)


torch.Size([8, 3424, 128])


In [38]:
transformer_layer = nn.TransformerEncoderLayer(
    d_model=128,
    nhead=8,
    dim_feedforward=256,
    dropout=0.1,
    batch_first=True
)

transformer = nn.TransformerEncoder(
    transformer_layer,
    num_layers=2
).to(device)


In [39]:
fused_tokens = transformer(tokens)

print(
    "Fused tokens:",
    fused_tokens.shape
)


Fused tokens: torch.Size([8, 3424, 128])


In [40]:
fused_feature = fused_tokens.mean(dim=1)

print(
    "Fused feature:",
    fused_feature.shape
)


Fused feature: torch.Size([8, 128])


In [41]:
control_head = nn.Sequential(
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
).to(device)


In [42]:
prediction = control_head(
    fused_feature
)

print("Prediction:", prediction.shape)


Prediction: torch.Size([8, 3])


In [ ]:
所以整條 pipeline 現在變成：

                 RGB
                  │
                  ▼
             CNN Encoder
                  │
                  ▼
           [8,128,20,120]
                  │
                  ▼
            2400 tokens
                  │
                  │
                  ├──────────┐
                  │          │
                  ▼          │
              Transformer   │
                  ▲          │
                  │          │
                  ├──────────┘
                  │
            1024 LiDAR tokens
                  ▲
                  │
             CNN Encoder
                  ▲
                  │
                LiDAR

最後：

3424 tokens
    ↓
Transformer
    ↓
mean pooling
    ↓
128 features
    ↓
Linear
    ↓
[steer, throttle, brake]

10. 但是這裡有一個很重要的問題
你現在看到：

3424 tokens

而 Transformer 的 self-attention 理論上需要處理：

3424 × 3424

的 attention matrix。

也就是：

3424² ≈ 11.7 million

而且你 batch 是：

8

所以 GPU memory 會開始變得很吃緊。

這就是為什麼真正的 TransFuser 不會這麼天真地直接把所有高解析度 token 全丟進 Transformer。

我們剛才這個版本是為了讓你理解：

CNN feature map → tokens → cross-modal attention → fused representation

而真正做模型時，我們需要降低 token 數量。

這次我們做真正的 Transformer Fusion
我們先不要直接照官方 TransFuser 的全部細節，而是做一個容易理解、可以在你的 Kaggle GPU 跑的版本。

目標：

RGB
[8,3,160,960]
       ↓
CNN
       ↓
[8,128,10,30]
       ↓
300 tokens
       │
       │
       ├──── Cross-modal Transformer ────┐
       │                                 │
LiDAR                                  Fusion
[8,3,256,256]                            │
       ↓                                 │
CNN                                      │
       ↓                                 │
[8,128,16,16]                            │
       ↓                                 │
256 tokens                               │
       │                                 │
       └─────────────────────────────────┘
                       ↓
                 fused tokens
                       ↓
                   pooling
                       ↓
               steer/throttle/brake

總共：

300 + 256 = 556 tokens

而不是：

2400 + 1024 = 3424 tokens

In [44]:
import torch
import torch.nn as nn


RGB Encoder

In [45]:
class RGBEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            # 160x960
            nn.Conv2d(
                3, 32,
                kernel_size=5,
                stride=2,
                padding=2
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # 80x480
            nn.Conv2d(
                32, 64,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # 40x240
            nn.Conv2d(
                64, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 20x120
            nn.Conv2d(
                128, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 10x60
            nn.Conv2d(
                128, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 5x30
            nn.Conv2d(
                128, 128,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 5x30 → 10x30
            # 保留較合理的長寬比例
            nn.AdaptiveAvgPool2d((10, 30))
        )

    def forward(self, x):
        return self.net(x)


2. LiDAR Encoder

In [46]:
class LiDAREncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            # 256x256
            nn.Conv2d(
                3, 32,
                kernel_size=5,
                stride=2,
                padding=2
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # 128x128
            nn.Conv2d(
                32, 64,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # 64x64
            nn.Conv2d(
                64, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 32x32
            nn.Conv2d(
                128, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 16x16
            nn.Conv2d(
                128, 128,
                kernel_size=3,
                stride=2,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


所以：

LiDAR
256×256
 ↓
128×128
 ↓
64×64
 ↓
32×32
 ↓
16×16

最後：

16 × 16 = 256 tokens

3. 建立 Encoder

In [47]:
rgb_encoder = RGBEncoder().to(device)
lidar_encoder = LiDAREncoder().to(device)


測試：

In [48]:
with torch.no_grad():

    rgb_feature = rgb_encoder(rgb)

    lidar_feature = lidar_encoder(lidar)

print("RGB feature:", rgb_feature.shape)
print("LiDAR feature:", lidar_feature.shape)


RGB feature: torch.Size([8, 128, 10, 30])
LiDAR feature: torch.Size([8, 128, 8, 8])


In [49]:
def feature_to_tokens(feature):

    B, C, H, W = feature.shape

    tokens = feature.flatten(2)

    tokens = tokens.transpose(1, 2)

    return tokens


In [50]:
rgb_tokens = feature_to_tokens(
    rgb_feature
)

lidar_tokens = feature_to_tokens(
    lidar_feature
)

print("RGB tokens:", rgb_tokens.shape)
print("LiDAR tokens:", lidar_tokens.shape)

RGB tokens: torch.Size([8, 300, 128])
LiDAR tokens: torch.Size([8, 64, 128])


6. 寫 Cross Attention

In [51]:
class CrossModalAttention(nn.Module):

    def __init__(
        self,
        embed_dim=128,
        num_heads=8
    ):
        super().__init__()

        self.rgb_to_lidar = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.lidar_to_rgb = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.rgb_norm = nn.LayerNorm(
            embed_dim
        )

        self.lidar_norm = nn.LayerNorm(
            embed_dim
        )

    def forward(
        self,
        rgb_tokens,
        lidar_tokens
    ):

        # RGB query
        rgb_attended, _ = self.rgb_to_lidar(
            query=rgb_tokens,
            key=lidar_tokens,
            value=lidar_tokens
        )

        rgb_tokens = self.rgb_norm(
            rgb_tokens + rgb_attended
        )


        # LiDAR query
        lidar_attended, _ = self.lidar_to_rgb(
            query=lidar_tokens,
            key=rgb_tokens,
            value=rgb_tokens
        )

        lidar_tokens = self.lidar_norm(
            lidar_tokens + lidar_attended
        )

        return rgb_tokens, lidar_tokens


7. RGB → LiDAR 是什麼意思？
這一行：

rgb_attended, _ = self.rgb_to_lidar(
    query=rgb_tokens,
    key=lidar_tokens,
    value=lidar_tokens
)

意思不是：

把 RGB 和 LiDAR 加起來。

而是：

每一個 RGB token 問：「LiDAR 裡面哪些資訊跟我有關？」

例如：

RGB token
「我看到一台車」

        ↓ attention

LiDAR tokens

「這裡有物體」
「這裡有道路」
「這裡有另一台車」
「這裡是空的」

Transformer 自己學：

RGB token
   ↓
attention weights
   ↓
LiDAR information

所以模型可以學到跨模態關聯。

8. 建立 Fusion

In [52]:
fusion = CrossModalAttention(
    embed_dim=128,
    num_heads=8
).to(device)


In [53]:
rgb_fused, lidar_fused = fusion(
    rgb_tokens,
    lidar_tokens
)


In [54]:
print(
    "RGB fused:",
    rgb_fused.shape
)

print(
    "LiDAR fused:",
    lidar_fused.shape
)


RGB fused: torch.Size([8, 300, 128])
LiDAR fused: torch.Size([8, 64, 128])


9. 最後做 Control Head

In [55]:
fused_tokens = torch.cat(
    [
        rgb_fused,
        lidar_fused
    ],
    dim=1
)

print(fused_tokens.shape)


torch.Size([8, 364, 128])


In [56]:
fused_feature = fused_tokens.mean(
    dim=1
)

print(fused_feature.shape)


torch.Size([8, 128])


In [57]:
control_head = nn.Sequential(

    nn.Linear(
        128,
        128
    ),

    nn.ReLU(),

    nn.Linear(
        128,
        3
    )
).to(device)


In [58]:
prediction = control_head(
    fused_feature
)

print(
    "Prediction:",
    prediction.shape
)


Prediction: torch.Size([8, 3])


10. 現在我們有真正的 Fusion Model 了
整個流程：

                   RGB
                    │
                    ▼
              RGB Encoder
                    │
                    ▼
              10 × 30 feature
                    │
                    ▼
               300 tokens
                    │
                    │ Query
                    ▼
              ┌───────────┐
              │ Cross     │
              │ Attention │
              └───────────┘
                    ▲
                    │ Key / Value
                    │
              256 LiDAR tokens
                    ▲
                    │
              LiDAR Encoder
                    ▲
                    │
                  LiDAR

而且我們不是只做：

RGB + LiDAR → concat

而是：

RGB token
    ↕
LiDAR token

互相透過 attention 交換資訊。



現在直接整理成完整 Class
把前面的東西合在一起：

In [59]:
import torch
import torch.nn as nn


class RGBEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(3, 32, 5, 2, 2),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((10, 30))
        )

    def forward(self, x):
        return self.net(x)


class LiDAREncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(3, 32, 5, 2, 2),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class CrossModalAttention(nn.Module):

    def __init__(
        self,
        embed_dim=128,
        num_heads=8
    ):
        super().__init__()

        self.rgb_to_lidar = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.lidar_to_rgb = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.rgb_norm = nn.LayerNorm(embed_dim)
        self.lidar_norm = nn.LayerNorm(embed_dim)

    def forward(
        self,
        rgb_tokens,
        lidar_tokens
    ):

        # RGB 看 LiDAR
        rgb_attended, _ = self.rgb_to_lidar(
            query=rgb_tokens,
            key=lidar_tokens,
            value=lidar_tokens
        )

        rgb_tokens = self.rgb_norm(
            rgb_tokens + rgb_attended
        )

        # LiDAR 看 RGB
        lidar_attended, _ = self.lidar_to_rgb(
            query=lidar_tokens,
            key=rgb_tokens,
            value=rgb_tokens
        )

        lidar_tokens = self.lidar_norm(
            lidar_tokens + lidar_attended
        )

        return rgb_tokens, lidar_tokens


class TransFuserPractice(nn.Module):

    def __init__(self):

        super().__init__()

        self.rgb_encoder = RGBEncoder()

        self.lidar_encoder = LiDAREncoder()

        self.fusion = CrossModalAttention(
            embed_dim=128,
            num_heads=8
        )

        self.head = nn.Sequential(

            nn.Linear(128, 128),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(128, 3)
        )


    def feature_to_tokens(self, feature):

        # [B,C,H,W]
        B, C, H, W = feature.shape

        # [B,C,H*W]
        feature = feature.flatten(2)

        # [B,H*W,C]
        feature = feature.transpose(1, 2)

        return feature


    def forward(
        self,
        rgb,
        lidar
    ):

        # =====================
        # RGB
        # =====================

        rgb_feature = self.rgb_encoder(rgb)

        rgb_tokens = self.feature_to_tokens(
            rgb_feature
        )


        # =====================
        # LiDAR
        # =====================

        lidar_feature = self.lidar_encoder(lidar)

        lidar_tokens = self.feature_to_tokens(
            lidar_feature
        )


        # =====================
        # Cross-modal Fusion
        # =====================

        rgb_tokens, lidar_tokens = self.fusion(
            rgb_tokens,
            lidar_tokens
        )


        # =====================
        # Combine
        # =====================

        fused_tokens = torch.cat(
            [
                rgb_tokens,
                lidar_tokens
            ],
            dim=1
        )


        # =====================
        # Global representation
        # =====================

        fused_feature = fused_tokens.mean(
            dim=1
        )


        # =====================
        # Control
        # =====================

        control = self.head(
            fused_feature
        )

        return control


建立 Model

In [60]:
model = TransFuserPractice().to(device)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

Parameters: 931331


In [61]:
prediction = model(
    rgb,
    lidar
)

print(
    "Prediction:",
    prediction.shape
)


Prediction: torch.Size([8, 3])


In [62]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


然後第一版先跑 3 epochs：

In [63]:
for epoch in range(3):

    model.train()

    total_loss = 0.0

    for batch in train_loader:

        rgb = batch["rgb"].to(
            device,
            non_blocking=True
        )

        lidar = batch["lidar"].to(
            device,
            non_blocking=True
        )

        target = batch["target"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        prediction = model(
            rgb,
            lidar
        )

        loss = criterion(
            prediction,
            target
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()


    avg_loss = (
        total_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch+1}/3 "
        f"Loss: {avg_loss:.6f}"
    )


Epoch 1/3 Loss: 0.066959
Epoch 2/3 Loss: 0.032174
Epoch 3/3 Loss: 0.019264


In [64]:
torch.save(
    model.state_dict(),
    "/kaggle/working/transfuser_practice.pth"
)

Step 1 下載官方 TransFuser GitHub
       ↓
Step 2 不下載 210GB dataset
       ↓
Step 3 只下載官方 pretrained weights
       ↓
Step 4 找到官方 model class
       ↓
Step 5 看懂官方 RGB/LiDAR preprocessing
       ↓
Step 6 嘗試 load checkpoint
       ↓
Step 7 用你的 Kaggle 小資料做 inference
       ↓
Step 8 再決定要不要 fine-tune

你可以做成這個實驗
                    同一份 Dataset
                          │
              ┌───────────┴───────────┐
              │                       │
              ▼                       ▼
       Pipeline A                Pipeline B
       我們自己做的                官方 TransFuser
              │                       │
              ▼                       ▼
        Train from scratch       Train from scratch
              │                       │
              ▼                       ▼
        Weight A                  Weight B
              │                       │
              └──────────┬────────────┘
                         ▼
                  同一份 Validation
                         │
                         ▼
                  比較 evaluation

但要注意「官方 pretrained weights」是另一個實驗
我們其實可以做 三組。

A：我們的模型，從零開始
你的 Dataset
     ↓
TransFuserPractice
     ↓
random initialization
     ↓
train
     ↓
Weight A

B：官方 TransFuser，從零開始
你的 Dataset
     ↓
Official TransFuser
     ↓
random initialization
     ↓
train
     ↓
Weight B

C：官方 TransFuser + 官方 pretrained weights
官方 pretrained weights
          ↓
Official TransFuser
          ↓
你的 Dataset
          ↓
fine-tune
          ↓
Weight C

這三個實驗就非常有意思。

為什麼 A vs B 特別重要？
因為如果：

A = 我們的簡化模型
B = 官方 TransFuser

都從 random initialization 開始，而且使用：

同一 train set
同一 validation set
同一 batch size
同一 optimizer
相近 learning rate
同樣 epoch
同樣 target

那你比較的是：

architecture / pipeline 本身的差異。

而不是：

「誰拿到比較好的 pretrained weights」。

C 則回答另一個問題
如果：

A：自己模型 + scratch
B：官方 TransFuser + scratch
C：官方 TransFuser + pretrained

可能得到：

                  Validation Loss
A                 0.0xx
B                 0.0xx
C                 0.0xx

我們就可以分析：

A vs B
架構差異

Transformer fusion 到底比我們簡單的 fusion 好多少？

B vs C
Pretraining 的效果

官方 pretrained knowledge 對我們這份資料有沒有幫助？

這兩個問題要分開，這樣才有研究價值。

而且不要只看 Loss
例如：

Model A
MSE = 0.019

Model B
MSE = 0.015

不能只說 B 比較好。

因為自駕控制最好再看：

Steer MAE
Throttle MAE
Brake accuracy

甚至之後如果我們把 CARLA 跑起來，可以比較：

Route completion
Collision
Infraction
Driving score

這才是比較完整的 evaluation。

所以我會建議我們最後做成這個實驗表
Experiment	Architecture	Initialization	Data
A	我們的 Practice	Random	相同
B	官方 TransFuser	Random	相同
C	官方 TransFuser	Pretrained	相同

然後：

               Train Loss
                   +
             Validation Loss
                   +
        Steer / Throttle / Brake
                   +
          CARLA evaluation

這樣你的練習就不只是：

「我成功跑了 TransFuser。」

而是變成一個很完整的實驗：

自己實作的 RGB-LiDAR fusion vs 官方 TransFuser vs 官方 pretrained TransFuser，在相同 CARLA 資料上的比較。

而且我們現在已經完成 A 的第一個版本，你的：

0.066959
→ 0.032174
→ 0.019264

就是第一筆 baseline。

下一步最合理的事情，就是把資料 split 固定下來，然後我幫你把官方 TransFuser pipeline 搬進 Kaggle，先做 B：官方架構、從零訓練。 之後再做 C 的 pretrained weights，這樣三組比較才公平。